# Generate Style Similarity Score
### Deprecated - use generate_train_data.ipynb
### This file creates the style similarity scores for training the multi-head cross-encoder
##### Created by w45242hy

## Install dependency

In [23]:
from pathlib import Path
root_path = Path().resolve().parent.parent
print(f"root_path: {root_path}")

!pip install -r {root_path}/requirements.txt

root_path: C:\Users\jacob\Data\UK\UoM\Year3\GitRepos\COMP34812-nlu-coursework



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import dependency

In [24]:
from dataset_av import AVDataset
import os
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, util

## Global variable

In [ ]:
ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))
CSV_NAME = f"AV_trial.csv"
CSV_PATH = os.path.join(ROOT_PATH, "data", "AV", CSV_NAME)
OUTPUT_CSV_NAME = f"{CSV_NAME.split('.')[0]}_style_similarity.{CSV_NAME.split('.')[1]}"
OUTPUT_CSV_PATH = os.path.join(ROOT_PATH, "data", "AV", OUTPUT_CSV_NAME)
MODEL_NAME = f"sentence-transformers/all-MiniLM-L6-v2"
SEPARATION_TOKEN = f"[SEP]"

## Generate style similarity score

### Load data

In [26]:
print(f"Reading csv file at {CSV_PATH} ...")
    
# Load dataset
dataset = AVDataset(CSV_PATH)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)  # Larger batch for efficiency

Reading csv file at c:\Users\jacob\Data\UK\UoM\Year3\GitRepos\COMP34812-nlu-coursework\data\AV\train.csv ...


### Load model

In [27]:
# Load pre-trained sentence transformer
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2901.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Generate style similarity score

In [28]:
similarity_scores = []
all_text1 = []
all_text2 = []

for texts, labels in dataloader:
    # texts is a list of concatenated strings; we need to split them
    for combined_text in texts:
        # Split back into text_1 and text_2
        text1, text2 = combined_text.split(SEPARATION_TOKEN)
        text1 = text1.strip()
        text2 = text2.strip()

        # Compute embeddings
        embedding_1 = model.encode(text1, convert_to_tensor=True)
        embedding_2 = model.encode(text2, convert_to_tensor=True)

        # Cosine similarity
        similarity_score = util.cos_sim(embedding_1, embedding_2).item()

        # Save
        similarity_scores.append(similarity_score)
        all_text1.append(text1)
        all_text2.append(text2)

### Create dataframe

In [29]:
# Create new DataFrame
df_output = pd.DataFrame({
    "text_1": all_text1,
    "text_2": all_text2,
    "similarity_score": similarity_scores
})

### Create output csv

In [30]:
# Save CSV
df_output.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Similarity scores saved to {OUTPUT_CSV_PATH}")

Similarity scores saved to c:\Users\jacob\Data\UK\UoM\Year3\GitRepos\COMP34812-nlu-coursework\data\AV\train_style_similarity.csv
